In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.5 Positional information

Attention is order-blind: by itself it sees a *set* of tokens, so "dog bites
man" and "man bites dog" would look identical. We add a position vector to each
token so order survives.

In [ ]:
from torch import nn

block, dim = 16, 8
pos_emb = nn.Embedding(block, dim)
p0 = pos_emb(torch.tensor(0))
p1 = pos_emb(torch.tensor(1))
print("position 0 and 1 get different vectors:", not torch.equal(p0, p1))

PocketLM (u4) adds these learned position vectors to the token embeddings.
u6 replaces them with RoPE, a smarter relative scheme.

In [ ]:
assert not torch.equal(p0, p1)